In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import holidays
import tkinter as tk
from tkinter import filedialog
from collections import defaultdict
from clase_reportes_new import ReportClassNew
import json
from collections import defaultdict
from send_mail import MailSender
import pandas as pd
ms = MailSender()
rc = ReportClassNew()
from drive_loader import DriveLoader, DRIVE_IDS
from db_loader import DBLoader
from bs4 import BeautifulSoup
import re


dl     = DriveLoader()
loader = DBLoader()



In [ ]:
# Leer archivos en el drive
archivos = dl.list_folder('1egufDljbBtYeMSmvDsyPk5-slIEl68nK')

odoo =[]
for f in archivos:
        df = dl.read_excel(f['id'])
        odoo.append(df)
        print(f"{f['name']} → {f['id']}")   
df_base = pd.concat(odoo, ignore_index=True)

# Lee el archivo desde el Drive usando el ID 
df_puc = dl.read_csv('1ra29Fdh4FF6TqObtHsl-pBpqJA1HNNII', sep=';', encoding='utf-8-sig')

df_cc = dl.read_excel('1BnHaFu0hF0OLpOFSjNaz3k2LaUOC-7V8', sheet_name='CC')

influencer =dl.read_excel('1BnHaFu0hF0OLpOFSjNaz3k2LaUOC-7V8', sheet_name='INFLUENCER')


In [ ]:

# Agregar validaciones
df_base = df_base.rename(columns={'Cuenta': 'Cuenta Origen'})


In [ ]:
df_base['Cuenta'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[0]
df_base['Nombre cuenta'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[1:].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')



In [ ]:
df_base['Clase'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[0].astype(str).str[0]

In [ ]:
df_base = df_base[df_base['Clase']!='n'].reset_index(drop=True)

In [ ]:
df_base['Clase'] = df_base['Clase'].astype(int)

In [ ]:

df_base['Grupo'] = df_base['Cuenta Origen'].astype(str).str[:2].astype(int)
df_base['Cuenta'] = df_base['Cuenta Origen'].astype(str).str[:4].astype(int)
df_base['Subcuenta'] = df_base['Cuenta Origen'].astype(str).str[:6].astype(int)

 # Convertir codigo a string para unificar tipos                                                                                                                                                                                           
df_puc['codigo'] = df_puc['codigo'].astype(int)


In [ ]:

# Mapas código → nombre por nivel
map_clase      = df_puc[df_puc['codigo']< 10].set_index('codigo')['descripcion']
map_grupo      = df_puc[(df_puc['codigo']>= 10) & (df_puc['codigo']< 100)].set_index('codigo')['descripcion']
map_cuenta     = df_puc[(df_puc['codigo']>= 100) & (df_puc['codigo']< 10000)].set_index('codigo')['descripcion']
map_subcuenta  = df_puc[(df_puc['codigo']>= 10000) & (df_puc['codigo']< 1000000)].set_index('codigo')['descripcion']


In [ ]:

# Agregar nombres a df_base
df_base['nombre_clase']     = df_base['Clase'].map(map_clase).fillna('Sin clase')
df_base['nombre_grupo']     = df_base['Grupo'].map(map_grupo).fillna('Sin grupo')
df_base['nombre_cuenta_jerarquia']    = df_base['Cuenta'].map(map_cuenta).fillna('Sin cuenta')
df_base['nombre_subcuenta'] = df_base['Subcuenta'].map(map_subcuenta).fillna('Sin subcuenta')


In [ ]:
def limpiar_html(texto):
    if pd.isna(texto):
        return texto
    soup = BeautifulSoup(texto, 'html.parser')
    texto_limpio = soup.get_text(separator=' ')
    texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()
    return texto_limpio

df_base['Asiento contable/Términos y condiciones'] = df_base['Asiento contable/Términos y condiciones'].apply(limpiar_html)

# Aqui

In [ ]:
canales_odoo = df_base['Líneas analíticas/Canal'].value_counts(dropna=True).reset_index()
canales_odoo['canal_odoo'] = canales_odoo['Líneas analíticas/Canal']
import unicodedata
canales_odoo['canal_procesado'] = canales_odoo['Líneas analíticas/Canal'].str.split(' ').str[1].str.upper().apply(lambda x: ''.join(c for c in unicodedata.normalize('NFD', x) if unicodedata.category(c) != 'Mn') if pd.notna(x) else x)
df_base['Asiento contable/Tipo de cliente'] = df_base['Asiento contable/Tipo de cliente'].str.upper()
canales_odoo['id'] = canales_odoo['canal_procesado'].str[:5]
df_base['id'] = df_base['Asiento contable/Tipo de cliente'].str[:5]

df_base_merge = df_base.merge(canales_odoo[['canal_odoo','canal_procesado', 'id']], on='id', how='left')

df_base_merge['canal_odoo'] = df_base_merge['canal_odoo'].fillna(df_base_merge['Asiento contable/Tipo de cliente'])


In [45]:
df_base_merge['canal_odoo'].value_counts()

canal_odoo
[CAN-MAY] Mayoristas         49387
[CF] Cliente Final           29979
[CAN-DIST] Distribuidores     9731
PROVEEDORES                   4254
[CAN-FARM] Farmacia            814
SURTICOSMETICOS                692
EMPLEADO                       668
ESPECIALIZADAS                 448
[CAN-CAT] Catálogo             365
KRIKA                          318
[CAN-COOP] Coopidrogas         283
EXTERIOR                       276
HOLE COSMETICS                 149
NEWSLETTER                      30
WROTE JUDGE.ME WEB REVIEW       15
Name: count, dtype: int64

In [71]:

contacto_tipo_c = df_base_merge[['Contacto','canal_odoo', 'NIT']].drop_duplicates()


In [72]:
contacto_tipo_c = contacto_tipo_c[~contacto_tipo_c['canal_odoo'].isna()]

In [67]:
contacto_tipo_c.shape

(2865, 3)

In [76]:
# Duplicados por Contacto y NIT
duplicados_contacto_tipo_c = contacto_tipo_c[contacto_tipo_c.duplicated(subset=['Contacto', 'NIT'], keep=False)]

In [77]:
contacto_tipo_c = contacto_tipo_c.drop_duplicates(subset=['Contacto' ,'NIT'],keep=False)

In [81]:
df_base_merge = df_base_merge.merge(contacto_tipo_c, on=['Contacto', 'NIT'], how='left', suffixes=('', '_contacto'))

In [85]:
df_base_merge['canal_odoo_contacto'] = df_base_merge['canal_odoo_contacto'].fillna(df_base_merge['Líneas analíticas/Canal'])

In [89]:
df_base_merge.to_csv(r'D:\desktop\df_base_merge.csv', index=False, sep=';', encoding='utf-8-sig')

In [88]:
df_base_merge.columns

Index(['Balance', 'Cantidad', 'Contacto', 'Creado el', 'Creado por',
       'Cuenta Origen', 'Diario', 'Empresa', 'Etiqueta', 'Fecha',
       'Fecha de factura', 'NIT', 'Número', 'Precisión analítica', 'Producto',
       'Referencia', 'Líneas analíticas/Canal',
       'Líneas analíticas/Centro de costos', 'Líneas analíticas/Cliente',
       'Líneas analíticas/HFAG', 'Líneas analíticas/Importe',
       'Líneas analíticas/La Poción', 'Líneas analíticas/Línea de Producto',
       'Líneas analíticas/País', 'Líneas analíticas/Proyectos',
       'Líneas analíticas/Shopify', 'Líneas analíticas/Surti',
       'Líneas analíticas/Tipo de Producto',
       'Asiento contable/Términos y condiciones',
       'Asiento contable/Tipo de cliente', 'Cuenta', 'Nombre cuenta', 'Clase',
       'Grupo', 'Subcuenta', 'nombre_clase', 'nombre_grupo',
       'nombre_cuenta_jerarquia', 'nombre_subcuenta', 'id', 'canal_odoo',
       'canal_procesado', 'canal_odoo_contacto'],
      dtype='object')

In [ ]:
# df_base.to_csv(r'D:\desktop\base_contable_completa.csv', index=False, sep=';', encoding='utf-8-sig')

In [ ]:

def extraer_clave(diccionario_str):
    if pd.isna(diccionario_str):
        return None
    try:
        diccionario = json.loads(diccionario_str)
        return list(diccionario.keys())[0]
    except Exception:
        return None

df_base = df_base.rename(columns={'Distribución analítica': 'Distribución analítica ori'})



In [ ]:

# Extraer el número del cc
df_base['Distribución analítica'] = df_base['Distribución analítica ori'].apply(
    lambda x: list(json.loads(x).keys())[0] if pd.notna(x) else None
)


In [ ]:

# Set las columnas previas
columnas = df_base.columns.tolist()
map_tipo = dict(zip(df_cc['cc'].astype(str), df_cc['TIPO']))
# Crea la nuevas columnas
df_base_merge = pd.concat([
    df_base,
    df_base['Distribución analítica']
        .apply(lambda x: {
            map_tipo.get(v.strip()): v.strip()
            for v in str(x).split(',')
            if map_tipo.get(v.strip())
        } if pd.notna(x) else {})
        .apply(pd.Series)
], axis=1)

df_base_merge['nombre_clase']     = df_base_merge['Clase'].map(map_clase).fillna('Sin clase')
df_base_merge['nombre_grupo']     = df_base_merge['Grupo'].map(map_grupo).fillna('Sin grupo')
df_base_merge['nombre_cuenta_jerarquia']    = df_base_merge['Cuenta'].map(map_cuenta).fillna('Sin cuenta')


# Ajustes manuales de asignación de centro de costo y concepto
df_base_merge['Clase'] = df_base_merge['Clase'].astype(str)
df_base_merge['Grupo'] = df_base_merge['Grupo'].astype(str)
df_base_merge['Cuenta'] = df_base_merge['Cuenta'].astype(str)


df_base_merge.loc[
    (df_base_merge['Cuenta'] == '4135'),
    'Centro de costos', 
] = '6'

df_base_merge.loc[
    (df_base_merge['Cuenta'] == '4175') & 
    (df_base_merge['Diario']!="Facturas de cliente Cali"),
    'Centro de costos', 
] = '6'

df_base_merge.loc[
    (df_base_merge['Clase'] == '6') & (df_base_merge['Centro de costos'].isna()),# revisar ##########
    'Centro de costos'
] =  '6'

df_base_merge.loc[
    (df_base_merge['Grupo'] == '42') & (df_base_merge['Centro de costos'].isna()),
    'Centro de costos'
] = '6'

df_base_merge.loc[(df_base_merge['Centro de costos'].isna()) & 
            (df_base_merge['Número'].str.startswith('BNK')) &
                (df_base_merge['Cuenta Origen'].isin(['530515001 COMISIONES','530505002 GRAVAMEN CUATRO POR MIL', '530505001 CUOTA DE MANEJO']))
            , 'Centro de costos'
            ] = '7'

df_base_merge.loc[(df_base_merge['Centro de costos'].isna()) & 
            (df_base_merge['Número'].str.startswith('BNK')) &
                (df_base_merge['Cuenta Origen'].isin(['539595001 AJUSTE A MILES']))
            , 'Centro de costos'
            ] = '6' 

df_base_merge.loc[(df_base_merge['Centro de costos'].isna()) & 
            (df_base_merge['Número'].str.startswith('STJ')) &
            (~df_base_merge['Contacto'].isin(influencer['Contacto'].unique().tolist()))
            , 'Centro de costos'
            ] = '6'  # validar si es clientre cc ==comercial  o infulerce cc== marketing ==

df_base_merge.loc[(df_base_merge['Centro de costos'].isna()) & 
            (df_base_merge['Número'].str.startswith('STJ')) &
            (df_base_merge['Contacto'].isin(influencer['Contacto'].unique().tolist()))
            , 'Centro de costos'
            ] = '4'  # validar si es clientre cc ==comercial  o infulerce cc== marketing ==

# Reemplaza los valores de las nuevas columnas
cols_nuevas = set(df_base_merge.columns) - set(columnas)

map_nombre = dict(
    zip(df_cc['cc'].astype(str), df_cc['Nombre Cencosto'])
)



for col in cols_nuevas:
    df_base_merge[col] = df_base_merge[col].map(map_nombre)

df_base_merge['Nombre Cencosto'] = df_base_merge['Centro de costos']
df_cc = df_cc[['Nombre Cencosto', 'Origen']].drop_duplicates()



df_base_merge = df_base_merge.merge(df_cc[['Nombre Cencosto', 'Origen' ]],
                                    left_on='Nombre Cencosto', right_on='Nombre Cencosto', how='left')



df_base_merge['Centro de costos']    = df_base_merge['Centro de costos'].fillna('Sin centro de costo')
                                                                                

df_base_merge['Nombre Cencosto']    = df_base_merge['Nombre Cencosto'].fillna('Sin centro de costo')



In [ ]:
df_base_merge['Clase'].value_counts()

In [ ]:
sin_centro_costo  = df_base_merge[(df_base_merge['Nombre Cencosto']=='Sin centro de costo')&(df_base_merge['Clase'].isin(['4', '5', '6']))]\
    .drop_duplicates(subset=['Número']) \
        [['Número', 'Cuenta Origen', 'Nombre cuenta', 'Diario', 'Contacto', 'Clase', 'Grupo', 'Cuenta', 'Subcuenta']] 

In [ ]:
df_base_merge.to_csv( r'D:\Desktop\base_contabilidad_prueba.csv', index=False, encoding='utf-8-sig', sep=';', decimal=',')

In [ ]:
sin_centro_costo.to_csv( r'D:\Desktop\sin_cc.csv', index=False, encoding='utf-8-sig', sep=';', decimal=',')

In [ ]:



# df_base_merge = df_base_merge.merge(df_concepto_unico, on='Cuenta', how='left')


# df_concepto['Nombre Cencosto'] = df_concepto['Nombre Cencosto'].str.upper().str.strip()

# df_base_merge['Nombre Cencosto'] = df_base_merge['Nombre Cencosto'].str.upper().str.strip()


# df_base_merge = df_base_merge.merge(df_concepto, on=['Cuenta','Nombre Cencosto' ], how='left')
# # Crea la columna conceto con base la los coceptos unicos y los que necesitan cc


# df_base_merge['Concepto_uni'] = df_base_merge['Concepto_uni'].fillna('Sin datos')
# df_base_merge['Concepto_cc'] = df_base_merge['Concepto_cc'].fillna('Sin datos')

# # df_base_merge['Concepto'] = np.where(df_base_merge['Concepto_uni'].isna(), df_base_merge['Concepto_cc'], df_base_merge['Concepto_uni'])
# df_base_merge = df_base_merge.reset_index(drop=True)
# df_base_merge['Concepto'] = np.where(
#     df_base_merge['Concepto_uni']=='Sin datos',
#     df_base_merge['Concepto_cc'],
#     df_base_merge['Concepto_uni']
# )
# Verifica las cuentas que no tienen concepto
# df_cuentas = df_base_merge[(df_base_merge['Concepto']=="Sin datos")&(df_base_merge['Nombre Cencosto'].notna())][['Cuenta','Nombre Cencosto', 'Nombre cuenta', 'Distribución analítica']]
# df_cuentas = df_cuentas.drop_duplicates(subset=['Cuenta', 'Nombre Cencosto',], keep='first')


In [ ]:
# Agregar validaciones

In [ ]:


return df_base_merge, df_cuentas


def archivos_contabilidad(self):
"""
Crea y consolida los archivos procesados por odoo

"""
ruta = self.validar_ruta()
ruta_contabilidad = ruta / 'data' / 'contabilidad'

df_base_merge, df_cuentas = self.contabilidad()

max_date = df_base_merge['Fecha'].max()
min_date = df_base_merge['Fecha'].min()
min_date.strftime('%d-%m-%Y')
ruta_base = ruta_contabilidad / 'base' / f'base_{min_date.strftime('%d-%m-%Y')}_{max_date.strftime('%d-%m-%Y')}.csv'
df_base_merge.to_csv(ruta_base, sep=";", index=False, encoding='utf-8', decimal=',')


centros_no_re = df_base_merge[(df_base_merge['Centro de costos'].isna())&
            (~df_base_merge['Distribución analítica'].isna())
            ][['Distribución analítica ori','Distribución analítica', 'Centro de costos']].drop_duplicates()
# Centros de costo mal clasificados
cc_corregir = df_base_merge[df_base_merge['Distribución analítica ori'].fillna('').str.count(':')>1]

# Genera el archivo de los casos sin centro de costos
sin_cc = df_base_merge[(df_base_merge['Nombre Cencosto'].isna() )]
sin_cc.to_excel(ruta_contabilidad / 'sin_cc.xlsx', index=False)


# Genera el archivo con los errores
with pd.ExcelWriter(ruta_contabilidad / 'correciones.xlsx', engine='openpyxl') as writer:
    sin_cc.to_excel(writer, index=False, sheet_name='Sin CC')
    cc_corregir.to_excel(writer, index=False, sheet_name='Corregir CC')
    centros_no_re.to_excel(writer, index=False, sheet_name='CC_no_registrados')
    df_cuentas.to_excel(writer, index=False, sheet_name='Cuentas sin concep')


# Crea un archivo para cada persona de contabilidad
digitadores = sin_cc['Creado por'].unique()
ruta_errores = ruta_contabilidad / 'correcciones'
dicc = {}
for i in digitadores:
    base = sin_cc[sin_cc['Creado por']==i]
    base.to_csv(ruta_errores / f'{i}.csv', index=False, sep=';', decimal=',', encoding='utf-8')
    dicc[i] = f'{i}.csv'
df_base_consol =  self.consolidar_carpeta(extension='csv', encoding='utf-8', sep=';', decimal=',', ruta_carpeta= ruta_contabilidad / 'base')
df_base_consol = df_base_consol.loc[:, ~df_base_consol.columns.str.contains('^Unnamed')]
df_base_consol.to_csv(ruta_contabilidad / 'base_consolidada.csv', encoding='utf-8', sep=';', decimal=',', index=False)
